# StatefulSets

Deployments verwalten austauschbare Pods; StatefulSets geben jedem Pod eine stabile nummerierte Identität und starten bzw. skalieren standardmäßig geordnet. Typische Anwendungen sind Datenbanken und verteilte Systeme. Persistenz ist ein separates Thema (Kapitel 11).

![headless-dns](headless-dns.svg)

## Voraussetzungen

Python-Kernel mit %%bash, kubectl und Zugriff auf den K3s-Trainingscluster. Notebook aus dem Kapitelverzeichnis ausführen. Beispielressourcen müssen frei sein; vorherige Übungen aufräumen. Aktiver Namespace muss default sein.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. StatefulSet und Headless Service

clusterIP: None bedeutet keine virtuelle Service-IP. DNS liefert die Adressen bereiter Pods. serviceName verknüpft das StatefulSet mit dem Headless Service.

In [ ]:
%%bash
set -eo pipefail
cat headless-svc.yml sts.yml

In [ ]:
%%bash
set -eo pipefail
kubectl apply -f headless-svc.yml -f sts.yml
kubectl rollout status statefulset/tkb-sts --timeout=180s
kubectl get pods -l app=web -o wide
kubectl get svc dullahan
kubectl run sts-toolbox --image=busybox:1.36 --restart=Never --command -- sleep 3600
kubectl wait --for=condition=Ready pod/sts-toolbox --timeout=180s

## 2. Individuelle DNS-Namen

Der Service-Name liefert mehrere Pod-Adressen; der individuelle Name adressiert genau einen Pod.

In [ ]:
%%bash
set -eo pipefail
kubectl exec sts-toolbox -- nslookup dullahan
for i in 0 1 2; do
    kubectl exec sts-toolbox -- wget -qO- http://tkb-sts-$i.dullahan/ | grep Hostname
done

## 3. Pod ersetzen

Die UID identifiziert das Objekt. Nach dem Löschen entsteht ein neuer Pod mit gleichem Namen, aber anderer UID.

In [ ]:
%%bash
set -eo pipefail
OLD=$(kubectl get pod tkb-sts-1 -o jsonpath='{.metadata.uid}')
echo "Alte UID: $OLD"
kubectl delete pod tkb-sts-1
for i in $(seq 1 60); do
    NEW=$(kubectl get pod tkb-sts-1 --ignore-not-found -o jsonpath='{.metadata.uid}')
    if [ -n "$NEW" ] && [ "$NEW" != "$OLD" ]; then break; fi
    sleep 2
done
test -n "$NEW" && test "$NEW" != "$OLD"
kubectl wait --for=condition=Ready pod/tkb-sts-1 --timeout=180s
echo "Neue UID: $NEW"

## 4. Skalierung

Beim Hochskalieren entstehen weitere nummerierte Pods; beim Herunterskalieren verschwinden die höchsten Ordinalzahlen zuerst.

In [ ]:
%%bash
set -eo pipefail
kubectl scale statefulset tkb-sts --replicas=5
kubectl rollout status statefulset/tkb-sts --timeout=180s
kubectl get pods -l app=web

In [ ]:
%%bash
set -eo pipefail
kubectl scale statefulset tkb-sts --replicas=2
kubectl rollout status statefulset/tkb-sts --timeout=180s
kubectl get pods -l app=web

## 5. Aufräumen

Cassandra und Cloud-Storage bleiben optionale Vertiefung.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f sts.yml -f headless-svc.yml
kubectl delete pod sts-toolbox
kubectl get pods -l app=web

## Fazit

StatefulSets bieten stabile Identitäten, aber nicht automatisch persistente Daten. Ein Headless Service ermöglicht individuelle DNS-Namen.